# BSI IT Service Desk AI Assistant
## Mini Project 5 - RAG Pipeline with Classification & Evaluation

**Architecture:**

| Step | Model | Function |
|------|-------|----------|
| 1. Classify | `gpt-4.1-nano` | Route ticket to category |
| 2. Retrieve | In-memory semantic search (cosine similarity) | Fetch relevant SOP chunks |
| 3. Generate | `gpt-4.1-mini` | Draft agent response with citations |
| 4. Evaluate | `gpt-4o` | Score groundedness, relevance, coherence |

**Data:**
- `bsi_ticket_dataset.csv` — helpdesk tickets (ticket_id, subject, sop_classification, priority, status)
- `BSI_IT_Service_Desk_Knowledge_Base.md` — SOP documents (VPN, MFA, Password Reset, SAP, Hardware, etc.)

## 0. Install Dependencies

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "openai", "numpy", "pandas", "tqdm", "-q"])
print("Dependencies installed.")

Dependencies installed.


## 1. Imports & Configuration

In [2]:
import os
import json
import time
import warnings
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

warnings.filterwarnings("ignore")
print("Libraries imported successfully.")

d:\Users\bsi80274\.conda\envs\ai-training\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported successfully.


In [ ]:
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "YOUR_OPENAI_API_KEY_HERE")

# Model configuration
CLASSIFIER_MODEL   = "gpt-4.1-nano"    
RESPONDER_MODEL    = "gpt-4.1-mini"    
EVALUATOR_MODEL    = "gpt-4o"          
EMBEDDING_MODEL    = "text-embedding-3-small"  

# Pipeline configuration
SEARCH_TOP_K       = 3                 
EVAL_SAMPLE_SIZE   = 20               

# File paths - adjust if your files are in a different directory
KB_FILE    = "BSI_IT_Service_Desk_Knowledge_Base__1_.md"
CSV_FILE   = "bsi_ticket_dataset.csv"
OUTPUT_CSV = "output_responses.csv"
EVAL_CSV   = "eval_report.csv"

# Cost estimation (USD per 1M tokens)
PRICING = {
    "gpt-4.1-nano":  {"input": 0.10,  "output": 0.40},
    "gpt-4.1-mini":  {"input": 0.40,  "output": 1.60},
    "gpt-4o":        {"input": 2.50,  "output": 10.00},
    "text-embedding-3-small": {"input": 0.02, "output": 0.0},
}

USD_TO_IDR = 16500

# SOP categories
CATEGORIES = ["password_reset", "vpn_issue", "erp_sap", "hardware_request",
               "software_request", "onboarding", "offboarding", "email_issue",
               "wifi_network", "mfa_reset", "out_of_scope"]


SOP_TO_CATEGORY = {
    "SOP-001": "vpn_issue",
    "SOP-002": "mfa_reset",
    "SOP-003": "password_reset",
    "SOP-004": "erp_sap",
    "SOP-005": "erp_sap",
    "SOP-006": "email_issue",
    "SOP-007": "hardware_request",
    "SOP-008": "hardware_request",
    "SOP-009": "software_request",
    "SOP-010": "onboarding",
    "SOP-011": "offboarding",
    "SOP-012": "email_issue",
    "SOP-013": "wifi_network",
}

print("Configuration loaded.")
print(f"  Classifier : {CLASSIFIER_MODEL}")
print(f"  Responder  : {RESPONDER_MODEL}")
print(f"  Evaluator  : {EVALUATOR_MODEL}")
print(f"  Embedding  : {EMBEDDING_MODEL}")
print(f"  Top-K      : {SEARCH_TOP_K}")

Configuration loaded.
  Classifier : gpt-4.1-nano
  Responder  : gpt-4.1-mini
  Evaluator  : gpt-4o
  Embedding  : text-embedding-3-small
  Top-K      : 3


In [9]:
# Auto-detect file paths (handles different working directories)
import os
import glob

def find_file(filename_pattern: str) -> str:
    
    search_dirs = [
        ".",                          # current working directory
        os.path.dirname(os.path.abspath("__file__")),  # notebook dir
        os.path.expanduser("~"),      # home directory
    ]
    for d in search_dirs:
        matches = glob.glob(os.path.join(d, filename_pattern))
        if matches:
            return matches[0]
    raise FileNotFoundError(
        f"File '{filename_pattern}' not found.\n"
        f"Searched in: {search_dirs}\n"
        f"Make sure the file is in the same folder as this notebook."
    )

KB_FILE  = find_file("D:/Users/bsi80274/OneDrive - PT. Berlian Sistem Informasi/AI-BOOTCAMP/TRAINING/hactiv8/sesi 37 - Mini Project 5 Deploy AI System on Azure/dataset/BSI_IT_Service_Desk_Knowledge_Base (1).md")
CSV_FILE = find_file("D:/Users/bsi80274/OneDrive - PT. Berlian Sistem Informasi/AI-BOOTCAMP/TRAINING/hactiv8/sesi 37 - Mini Project 5 Deploy AI System on Azure/dataset/bsi_ticket_dataset.csv")

print(f"KB file  found: {KB_FILE}")
print(f"CSV file found: {CSV_FILE}")

KB file  found: D:/Users/bsi80274/OneDrive - PT. Berlian Sistem Informasi/AI-BOOTCAMP/TRAINING/hactiv8/sesi 37 - Mini Project 5 Deploy AI System on Azure/dataset/BSI_IT_Service_Desk_Knowledge_Base (1).md
CSV file found: D:/Users/bsi80274/OneDrive - PT. Berlian Sistem Informasi/AI-BOOTCAMP/TRAINING/hactiv8/sesi 37 - Mini Project 5 Deploy AI System on Azure/dataset/bsi_ticket_dataset.csv


## 2. Initialize OpenAI Client

In [ ]:
client = OpenAI(api_key= 'OPENAI_API_KEY')


try:
    test = client.chat.completions.create(
        model=CLASSIFIER_MODEL,
        messages=[{"role": "user", "content": "reply: ok"}],
        max_tokens=5
    )
    print(f"OpenAI client connected. Test response: {test.choices[0].message.content.strip()}")
except Exception as e:
    print(f"ERROR connecting to OpenAI: {e}")
    print("Make sure OPENAI_API_KEY is set correctly.")

OpenAI client connected. Test response: OK


In [11]:
def estimate_cost(model: str, input_tokens: int, output_tokens: int) -> dict:
    price = PRICING.get(model, {"input": 0, "output": 0})
    cost_usd = (input_tokens / 1_000_000) * price["input"] + (output_tokens / 1_000_000) * price["output"]
    return {
        "model": model,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "cost_usd": round(cost_usd, 8),
        "cost_idr": round(cost_usd * USD_TO_IDR, 4),
    }

## 3. Load & Parse Knowledge Base (SOP Documents)

In [13]:
def load_knowledge_base(filepath: str) -> str:
    with open(filepath, "r", encoding="utf-8") as f:
        return f.read()

def chunk_document(text: str, source_path: str, chunk_size: int = 500, overlap: int = 50) -> list:
    chunks = []
    sections = re.split(r'(?=^## SOP-)', text, flags=re.MULTILINE)
    for section in sections:
        if not section.strip():
            continue
        sop_match = re.match(r'## (SOP-\d+)', section.strip())
        sop_id = sop_match.group(1) if sop_match else "SOP-GENERAL"
        words = section.split()
        words_per_chunk = chunk_size * 4 // 5
        overlap_words   = max(1, overlap * 4 // 5)
        i = 0
        while i < len(words):
            chunk_text = " ".join(words[i: i + words_per_chunk])
            chunks.append({"sop_id": sop_id, "content": chunk_text, "source": source_path})
            i += words_per_chunk - overlap_words
    return chunks

kb_text    = load_knowledge_base(KB_FILE)
sop_chunks = chunk_document(kb_text, source_path=KB_FILE, chunk_size=500, overlap=50)

print(f"Knowledge base loaded: {len(kb_text):,} characters")
print(f"Total chunks created : {len(sop_chunks)}")

from collections import Counter
sop_dist = Counter(c["sop_id"] for c in sop_chunks)
print("\nChunks per SOP:")
for sop, count in sorted(sop_dist.items()):
    print(f"  {sop}: {count} chunks")

Knowledge base loaded: 24,535 characters
Total chunks created : 15

Chunks per SOP:
  SOP-001: 2 chunks
  SOP-002: 1 chunks
  SOP-003: 1 chunks
  SOP-004: 1 chunks
  SOP-005: 1 chunks
  SOP-006: 1 chunks
  SOP-007: 1 chunks
  SOP-008: 1 chunks
  SOP-009: 1 chunks
  SOP-010: 1 chunks
  SOP-011: 1 chunks
  SOP-012: 1 chunks
  SOP-013: 1 chunks
  SOP-GENERAL: 1 chunks


## 4. Build In-Memory Vector Index (Semantic Retrieval)

In [14]:
def get_embedding(text: str) -> list:
    
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text,
    )
    return response.data[0].embedding


def cosine_similarity(a: list, b: list) -> float:
    
    a = np.array(a)
    b = np.array(b)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


print("Generating embeddings for all SOP chunks...")
print(f"This will embed {len(sop_chunks)} chunks using {EMBEDDING_MODEL}")

chunk_embeddings = []
batch_size = 20
total_embed_tokens = 0

for i in tqdm(range(0, len(sop_chunks), batch_size), desc="Embedding chunks"):
    batch = sop_chunks[i: i + batch_size]
    texts = [c["content"] for c in batch]
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    total_embed_tokens += response.usage.total_tokens
    for item in response.data:
        chunk_embeddings.append(item.embedding)
    time.sleep(0.1)

embed_cost = estimate_cost(EMBEDDING_MODEL, total_embed_tokens, 0)
print(f"\nEmbedding complete.")
print(f"  Total chunks embedded : {len(chunk_embeddings)}")
print(f"  Total tokens used     : {total_embed_tokens:,}")
print(f"  Estimated cost        : ${embed_cost['cost_usd']:.6f} (Rp {embed_cost['cost_idr']:.4f})")

Generating embeddings for all SOP chunks...
This will embed 15 chunks using text-embedding-3-small


Embedding chunks: 100%|██████████| 1/1 [00:03<00:00,  3.08s/it]


Embedding complete.
  Total chunks embedded : 15
  Total tokens used     : 7,363
  Estimated cost        : $0.000147 (Rp 2.4298)


In [15]:
def retrieve_sop_chunks(query: str, top_k: int = SEARCH_TOP_K) -> list:
    
    query_embedding = get_embedding(query)
    scores = [
        cosine_similarity(query_embedding, chunk_emb)
        for chunk_emb in chunk_embeddings
    ]

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_indices:
        chunk = sop_chunks[idx].copy()
        chunk["score"] = round(scores[idx], 4)
        results.append(chunk)

    return results


# Test retrieval
test_query = "How do I reset my MFA token on a new phone?"
test_chunks = retrieve_sop_chunks(test_query)

print(f"Test retrieval: '{test_query}'")
print(f"Retrieved {len(test_chunks)} chunk(s):\n")
for i, c in enumerate(test_chunks, 1):
    print(f"--- Chunk {i} | {c['sop_id']} | score={c['score']} ---")
    print(c["content"][:300], "...\n")

Test retrieval: 'How do I reset my MFA token on a new phone?'
Retrieved 3 chunk(s):

--- Chunk 1 | SOP-002 | score=0.6008 ---
## SOP-002: RESET TOKEN MFA (MULTI-FACTOR AUTHENTICATION) ### 1. Tujuan dan Ruang Lingkup Mengatur prosedur verifikasi dan reset token MFA (Microsoft Authenticator) untuk pengguna yang kehilangan ponsel, mengganti nomor HP, atau tidak sengaja menghapus aplikasi Authenticator. Keamanan sangat ketat d ...

--- Chunk 2 | SOP-003 | score=0.3459 ---
## SOP-003: ACTIVE DIRECTORY PASSWORD RESET ### 1. Tujuan dan Ruang Lingkup Dokumen ini menjadi acuan bagi agen Service Desk dalam menangani tiket *Password Reset* dan *Account Unlock* untuk domain `mitsubishi.local` dan Entra ID. ### 2. Kebijakan Kata Sandi (Password Policy) - Minimal 12 karakter.  ...

--- Chunk 3 | SOP-010 | score=0.2737 ---
## SOP-010: NEW EMPLOYEE ONBOARDING (IT PROVISIONING) ### 1. Trigger Otomatis Proses *onboarding* IT idealnya dipicu oleh integrasi HR (Workday) ke Active Directory, di mana akun t

## 5. Step 1 - Ticket Classifier (gpt-4.1-nano)

In [16]:
CLASSIFIER_SYSTEM = """You are an IT helpdesk ticket classifier for BSI (Binakarya Sarana Indonesia), 
supporting Mitsubishi Indonesia subsidiaries.

Classify the ticket into EXACTLY ONE category:
  - password_reset    : AD/SAP/email password reset or account unlock
  - vpn_issue         : VPN connection problems, FortiClient issues
  - erp_sap           : SAP login issues, SAP transaction errors, SAP authorization (SU53)
  - hardware_request  : Laptop, printer, scanner, peripheral requests or replacements
  - software_request  : Software installation, licensing requests (Visio, AutoCAD, etc.)
  - onboarding        : New employee IT setup, account creation
  - offboarding       : Employee account termination, resign, immediate offboarding
  - email_issue       : Outlook, Exchange, Teams, M365 issues, shared mailbox
  - wifi_network      : Corporate WiFi, 802.1x, guest WiFi access
  - mfa_reset         : MFA/Authenticator reset, new phone setup
  - out_of_scope      : Non-IT topics (HR, finance, canteen, etc.)

Rules:
- Reply ONLY with the category name, nothing else.
- If uncertain, pick the closest IT category rather than out_of_scope.

Examples:
  "Password SAP expired, cannot login"                  -> password_reset
  "VPN error 619 when connecting from home"             -> vpn_issue
  "SAP SU53 authorization error on ME21N"               -> erp_sap
  "Need new laptop for new finance staff"               -> hardware_request
  "Install MS Visio for project work"                   -> software_request
  "New hire onboarding, need to create AD account"      -> onboarding
  "Employee resigned, please disable account"          -> offboarding
  "Cannot receive MFA prompt on phone"                  -> mfa_reset
  "Cannot connect to MITSUBISHI-CORP wifi"              -> wifi_network
  "Outlook showing disconnected status"                 -> email_issue
  "When is the office holiday schedule?"                -> out_of_scope"""

OUT_OF_SCOPE_REPLY = (
    "Thank you for contacting BSI IT Helpdesk. Your inquiry appears to be outside the scope "
    "of IT support services. Please direct your question to the appropriate department (HR, Finance, Facilities, etc.). "
    "If you believe this is an IT issue, please resubmit with more details about the technical problem."
)


def classify_ticket(ticket_text: str) -> tuple:
    """
    Classify ticket using gpt-4.1-nano.
    Returns (category_string, cost_dict).
    """
    try:
        resp = client.chat.completions.create(
            model=CLASSIFIER_MODEL,
            messages=[
                {"role": "system", "content": CLASSIFIER_SYSTEM},
                {"role": "user",   "content": ticket_text},
            ],
            temperature=0,
            max_tokens=20,
        )
        raw = resp.choices[0].message.content.strip().lower().replace("-", "_").replace(" ", "_")
        cost = estimate_cost(CLASSIFIER_MODEL, resp.usage.prompt_tokens, resp.usage.completion_tokens)

        # Exact match
        if raw in CATEGORIES:
            return raw, cost
        # Fuzzy match
        for cat in CATEGORIES:
            if cat in raw:
                return cat, cost
        return "out_of_scope", cost

    except Exception as e:
        zero_cost = estimate_cost(CLASSIFIER_MODEL, 0, 0)
        return f"error:{e}", zero_cost


# Classifier smoke test
test_cases = [
    "Password SAP saya expired, tidak bisa login ke sistem",
    "VPN tidak bisa connect, error 619 terus muncul",
    "SAP SU53 authorization error saat buka transaksi ME21N",
    "Butuh laptop baru untuk karyawan baru divisi finance",
    "Install AutoCAD untuk tim engineering",
    "Onboarding karyawan baru, perlu buat akun AD",
    "Karyawan resign, tolong disable akunnya",
    "Ganti HP baru, perlu setup ulang MFA",
    "Tidak bisa connect ke wifi MITSUBISHI-CORP",
    "Outlook menunjukan status Disconnected",
    "Kapan jadwal cuti bersama tahun ini?",
]

print("=== Classifier Test ===")
print(f"{'Ticket Text':<55} {'Predicted Category':<20}")
print("-" * 80)
for t in test_cases:
    cat, _ = classify_ticket(t)
    print(f"{t[:55]:<55} {cat:<20}")

=== Classifier Test ===
Ticket Text                                             Predicted Category  
--------------------------------------------------------------------------------
Password SAP saya expired, tidak bisa login ke sistem   password_reset      
VPN tidak bisa connect, error 619 terus muncul          vpn_issue           
SAP SU53 authorization error saat buka transaksi ME21N  erp_sap             
Butuh laptop baru untuk karyawan baru divisi finance    hardware_request    
Install AutoCAD untuk tim engineering                   software_request    
Onboarding karyawan baru, perlu buat akun AD            onboarding          
Karyawan resign, tolong disable akunnya                 offboarding         
Ganti HP baru, perlu setup ulang MFA                    mfa_reset           
Tidak bisa connect ke wifi MITSUBISHI-CORP              wifi_network        
Outlook menunjukan status Disconnected                  email_issue         
Kapan jadwal cuti bersama tahun ini?            

## 6. Step 2 - Retrieve SOP Chunks (Semantic Search)

In [17]:
# Test retrieval with a realistic ticket
sample_ticket = "Locked out of SAP P01 after too many failed login attempts"
sample_category = "erp_sap"

# Enrich query with category context for better retrieval
enriched_query = f"{sample_ticket} {sample_category.replace('_', ' ')}"
chunks = retrieve_sop_chunks(enriched_query, top_k=SEARCH_TOP_K)

print(f"Ticket: '{sample_ticket}'")
print(f"Category: {sample_category}")
print(f"Retrieved {len(chunks)} chunk(s):\n")
for i, c in enumerate(chunks, 1):
    print(f"--- Chunk {i} | {c['sop_id']} | similarity={c['score']} ---")
    print(c["content"][:400], "...\n")

Ticket: 'Locked out of SAP P01 after too many failed login attempts'
Category: erp_sap
Retrieved 3 chunk(s):

--- Chunk 1 | SOP-004 | similarity=0.6592 ---
## SOP-004: SAP GUI LOGIN ISSUES & ACCOUNT UNLOCK ### 1. Tujuan Membantu pengguna yang mengalami kegagalan login ke aplikasi SAP ERP (SAP Logon / SAP GUI). Catatan penting: Sistem autentikasi SAP di Mitsubishi Indonesia saat ini **belum** terintegrasi penuh dengan SSO (Single Sign-On) Active Directory untuk semua sistem (Production vs QA/Dev). ### 2. Identifikasi Masalah Utama Tanyakan kepada peng ...

--- Chunk 2 | SOP-005 | similarity=0.5394 ---
## SOP-005: SAP TRANSACTION ERRORS & AUTHORIZATION (SU53) ### 1. Tujuan Menangani tiket pengguna yang mengeluh tidak dapat mengakses menu tertentu di SAP (contoh: tidak bisa buat PO di ME21N, tidak bisa lihat laporan keuangan di FBL3N). ### 2. SLA (Service Level Agreement) - **Prioritas:** Medium - **Waktu Penyelesaian:** 1 - 2 Hari Kerja (karena butuh approval pemilik bisnis/Business Proc

## 7. Step 3 - Generate Response (gpt-4.1-mini)

In [18]:
RESPONDER_SYSTEM = """You are a professional IT helpdesk agent at BSI (Binakarya Sarana Indonesia), 
supporting employees of Mitsubishi Indonesia subsidiaries.

Your task: Write a clear, professional support response for the IT ticket based on the provided SOP excerpts.

Guidelines:
- Write in Bahasa Indonesia.
- Use numbered steps for procedures.
- Use **bold** for important terms, system names, and commands.
- Mention the correct escalation team when needed.
- Begin with a polite greeting, end with an offer for further help.
- ONLY include steps that exist in the provided SOP excerpts. Do not invent procedures.
- If the SOP context is insufficient, say: "Tiket ini memerlukan penanganan lebih lanjut oleh tim IT Level 2."
- MANDATORY: End every response with a citation section: "[Sumber SOP: <SOP-ID list>]"""


def generate_response(ticket_text: str, category: str, chunks: list) -> tuple:
    
    if not chunks:
        sop_context = "Tidak ada dokumen SOP relevan yang ditemukan. Eskalasikan ke tim IT Level 2."
        sources = []
    else:
        sop_context = "\n\n---\n\n".join(
            f"[{c['sop_id']} - Chunk {i+1}]\n{c['content']}"
            for i, c in enumerate(chunks)
        )
        sources = list(dict.fromkeys(c["sop_id"] for c in chunks))

    user_prompt = f"""## IT Support Ticket
Category: {category.replace('_', ' ').title()}

Ticket description:
{ticket_text}

## Relevant SOP Excerpts
{sop_context}

## Task
Write a professional agent response based strictly on the SOP excerpts above.
At the end, include: [Sumber SOP: {', '.join(sources) if sources else 'N/A'}]"""

    try:
        resp = client.chat.completions.create(
            model=RESPONDER_MODEL,
            messages=[
                {"role": "system", "content": RESPONDER_SYSTEM},
                {"role": "user",   "content": user_prompt},
            ],
            temperature=0.3,
            max_tokens=900,
        )
        response_text = resp.choices[0].message.content.strip()
        cost = estimate_cost(RESPONDER_MODEL, resp.usage.prompt_tokens, resp.usage.completion_tokens)

        # Fallback: ensure citation is present
        if "Sumber SOP" not in response_text and sources:
            response_text += f"\n\n[Sumber SOP: {', '.join(sources)}]"

        return response_text, cost

    except Exception as e:
        zero_cost = estimate_cost(RESPONDER_MODEL, 0, 0)
        return f"[GENERATION ERROR: {e}]", zero_cost


# Test generation
draft, gen_cost = generate_response(sample_ticket, sample_category, chunks)
print("=" * 60)
print("GENERATED RESPONSE:")
print("=" * 60)
print(draft)
print()
print(f"Generation cost: ${gen_cost['cost_usd']:.6f} (Rp {gen_cost['cost_idr']:.4f})")

GENERATED RESPONSE:
Yth. Bapak/Ibu,

Terima kasih telah menghubungi IT Helpdesk BSI. Mengenai kendala terkunci akun SAP P01 akibat terlalu banyak percobaan login yang gagal, berikut langkah yang dapat kami lakukan untuk membantu:

1. Kami akan membuka aplikasi **SAP Logon** dan masuk ke sistem P01 (Production).
2. Jalankan T-Code: **SU01**.
3. Masukkan SAP User ID Anda.
4. Klik ikon **Lock/Unlock** (gambar gembok) untuk memeriksa status akun.
5. Jika akun terkunci karena *too many failed attempts*, kami akan klik tombol **Unlock** untuk membuka kunci akun Anda.
6. Setelah itu, silakan coba login kembali menggunakan password lama Anda.

Apabila Anda juga ingin melakukan reset password SAP, kami dapat membantu dengan prosedur berikut:

1. Jalankan T-Code: **SU01**.
2. Masukkan SAP User ID.
3. Klik ikon **Logon Data / Password** (gambar kunci).
4. Masukkan password standar awal: **Init123456!**.
5. Klik **Save**.
6. Setelah reset, SAP akan meminta Anda untuk membuat password baru saat log

## 8. Full Pipeline - End-to-End

In [19]:
def run_pipeline(ticket_id: str, ticket_text: str, verbose: bool = False) -> dict:
    
    t_total = time.perf_counter()
    cost_total_usd = 0.0

    # STEP 1: Classify
    t0 = time.perf_counter()
    category, cost_classify = classify_ticket(ticket_text)
    t_classify = round(time.perf_counter() - t0, 3)
    cost_total_usd += cost_classify["cost_usd"]

    # Handle out-of-scope
    if category == "out_of_scope" or category.startswith("error:"):
        return {
            "ticket_id": ticket_id,
            "ticket_text": ticket_text,
            "predicted_category": category,
            "retrieved_sources": "",
            "draft_response": OUT_OF_SCOPE_REPLY if category == "out_of_scope" else f"[PIPELINE ERROR: {category}]",
            "chunks_retrieved": 0,
            "latency_classify_s": t_classify,
            "latency_retrieve_s": 0.0,
            "latency_generate_s": 0.0,
            "latency_total_s": round(time.perf_counter() - t_total, 3),
            "cost_classify_usd": cost_classify["cost_usd"],
            "cost_generate_usd": 0.0,
            "cost_total_usd": cost_classify["cost_usd"],
            "cost_total_idr": round(cost_classify["cost_usd"] * USD_TO_IDR, 4),
        }

    # STEP 2: Retrieve
    t0 = time.perf_counter()
    enriched_query = f"{ticket_text} {category.replace('_', ' ')}"
    retrieved_chunks = retrieve_sop_chunks(enriched_query, top_k=SEARCH_TOP_K)
    t_retrieve = round(time.perf_counter() - t0, 3)

    # STEP 3: Generate
    t0 = time.perf_counter()
    draft, cost_generate = generate_response(ticket_text, category, retrieved_chunks)
    t_generate = round(time.perf_counter() - t0, 3)
    cost_total_usd += cost_generate["cost_usd"]

    if verbose:
        print(f"  Category  : {category} ({t_classify}s)")
        print(f"  Chunks    : {len(retrieved_chunks)} retrieved ({t_retrieve}s)")
        print(f"  Response  : {len(draft)} chars ({t_generate}s)")
        print(f"  Cost      : ${cost_total_usd:.6f} / Rp {cost_total_usd * USD_TO_IDR:.4f}")

    return {
        "ticket_id": ticket_id,
        "ticket_text": ticket_text,
        "predicted_category": category,
        "retrieved_sources": " | ".join(c["sop_id"] for c in retrieved_chunks),
        "draft_response": draft,
        "chunks_retrieved": len(retrieved_chunks),
        "latency_classify_s": t_classify,
        "latency_retrieve_s": t_retrieve,
        "latency_generate_s": t_generate,
        "latency_total_s": round(time.perf_counter() - t_total, 3),
        "cost_classify_usd": cost_classify["cost_usd"],
        "cost_generate_usd": cost_generate["cost_usd"],
        "cost_total_usd": round(cost_total_usd, 8),
        "cost_total_idr": round(cost_total_usd * USD_TO_IDR, 4),
    }


# Demo - single ticket
demo = run_pipeline(
    ticket_id="DEMO-001",
    ticket_text="VPN tidak bisa connect dari rumah. Muncul error credential or SSL VPN configuration wrong. Sudah coba restart tapi tetap tidak bisa.",
    verbose=True,
)
print("\n" + "=" * 60)
print("DRAFT RESPONSE:")
print("=" * 60)
print(demo["draft_response"])

  Category  : vpn_issue (0.656s)
  Chunks    : 3 retrieved (0.343s)
  Response  : 1516 chars (7.311s)
  Cost      : $0.001293 / Rp 21.3427

DRAFT RESPONSE:
Yth. Bapak/Ibu,

Terima kasih telah menghubungi IT Helpdesk BSI terkait kendala VPN yang tidak bisa terkoneksi dengan muncul error "credential or SSL VPN configuration wrong". Berikut langkah-langkah yang dapat kami sarankan untuk mengatasi masalah tersebut:

1. Verifikasi status akun Active Directory (AD) Anda melalui sistem kami, pastikan akun tidak terkunci (Locked Out). Jika terkunci, kami akan membuka kunci tersebut.
2. Pastikan akun Anda sudah terdaftar dalam grup keamanan `VPN-Remote-Access` di Entra ID. Jika belum, kami akan memerlukan persetujuan manajer Anda untuk menambahkan ke grup tersebut.
3. Jika sudah sesuai namun masalah tetap muncul, pastikan Anda menggunakan konfigurasi VPN FortiClient yang benar:
   - Connection Name: `Mitsubishi Corp VPN`
   - Remote Gateway: `vpn.mitsubishi.co.id`
   - Port: `443` (atau alterna

In [20]:
# Demo - out of scope ticket
demo_oos = run_pipeline(
    ticket_id="DEMO-002",
    ticket_text="Kapan jadwal cuti bersama lebaran tahun ini?",
    verbose=True,
)
print("\n" + "=" * 60)
print("DRAFT RESPONSE (Out of Scope):")
print("=" * 60)
print(demo_oos["draft_response"])


DRAFT RESPONSE (Out of Scope):
Thank you for contacting BSI IT Helpdesk. Your inquiry appears to be outside the scope of IT support services. Please direct your question to the appropriate department (HR, Finance, Facilities, etc.). If you believe this is an IT issue, please resubmit with more details about the technical problem.


## 9. Batch Processing - Process All Tickets from CSV

In [21]:
# Load ticket dataset
df = pd.read_csv(CSV_FILE)
print(f"Loaded {len(df)} tickets from {CSV_FILE}")
print()
print(df.head(10).to_string(index=False))
print()
print("Column names:", list(df.columns))
print()
print("SOP Classification distribution:")
print(df["sop_classification"].value_counts().to_string())

Loaded 150 tickets from D:/Users/bsi80274/OneDrive - PT. Berlian Sistem Informasi/AI-BOOTCAMP/TRAINING/hactiv8/sesi 37 - Mini Project 5 Deploy AI System on Azure/dataset/bsi_ticket_dataset.csv

    ticket_id                             subject sop_classification priority      status
TKT-2605-0001       Immediate offboarding request            SOP-011     High    Resolved
TKT-2605-0002               Locked out of SAP P01            SOP-004     High        Open
TKT-2605-0003           Convert to shared mailbox            SOP-011   Medium        Open
TKT-2605-0004      Account creation for new staff            SOP-010     High        Open
TKT-2605-0005               Locked out of SAP P01            SOP-004     High        Open
TKT-2605-0006        Install MS Visio for project            SOP-009      Low      Closed
TKT-2605-0007        AutoCAD installation request            SOP-009   Medium      Closed
TKT-2605-0008      SU53 authorization error ME21N            SOP-005   Medium In Progr

In [22]:
results = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing tickets"):
    ticket_id   = str(row.get("ticket_id", "N/A"))
    # Use 'subject' as the ticket text (this dataset uses subject instead of free-text)
    ticket_text = str(row.get("subject", "")).strip()
    true_sop    = str(row.get("sop_classification", "")).strip()
    true_cat    = SOP_TO_CATEGORY.get(true_sop, "unknown")

    try:
        result = run_pipeline(ticket_id, ticket_text)
        result["true_sop"]        = true_sop
        result["true_category"]   = true_cat
        result["priority"]        = str(row.get("priority", ""))
        result["status"]          = str(row.get("status", ""))
        result["correct"]         = int(true_cat == result["predicted_category"])
        result["error"]           = ""
    except Exception as e:
        result = {
            "ticket_id": ticket_id,
            "ticket_text": ticket_text,
            "true_sop": true_sop,
            "true_category": true_cat,
            "predicted_category": "error",
            "retrieved_sources": "",
            "draft_response": "",
            "correct": 0,
            "error": str(e),
        }

    results.append(result)
    time.sleep(0.2)  # Avoid rate limiting


# Save results
out_df = pd.DataFrame(results)
out_df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved {len(out_df)} results to {OUTPUT_CSV}")

Processing tickets: 100%|██████████| 150/150 [15:16<00:00,  6.11s/it]


Saved 150 results to output_responses.csv


In [23]:
# Batch processing summary
print("=" * 60)
print("BATCH PROCESSING SUMMARY")
print("=" * 60)

total_tickets = len(out_df)
error_count   = out_df["predicted_category"].str.startswith("error").sum()
valid_df      = out_df[~out_df["predicted_category"].str.startswith("error", na=False)]

# Classification accuracy (only on tickets with known true category)
known_df = valid_df[valid_df["true_category"] != "unknown"]
accuracy = known_df["correct"].mean() * 100 if len(known_df) > 0 else 0

# Latency stats
if "latency_total_s" in valid_df.columns:
    lat_p50 = valid_df["latency_total_s"].quantile(0.50)
    lat_p95 = valid_df["latency_total_s"].quantile(0.95)
else:
    lat_p50 = lat_p95 = 0

# Cost stats
if "cost_total_usd" in valid_df.columns:
    total_cost_usd = valid_df["cost_total_usd"].sum()
    avg_cost_usd   = valid_df["cost_total_usd"].mean()
else:
    total_cost_usd = avg_cost_usd = 0

print(f"Total tickets processed  : {total_tickets}")
print(f"Errors                   : {error_count}")
print(f"Classification accuracy  : {accuracy:.1f}% (on {len(known_df)} labeled tickets)")
print(f"Latency P50              : {lat_p50:.2f}s")
print(f"Latency P95              : {lat_p95:.2f}s")
print(f"Total cost (all tickets) : ${total_cost_usd:.6f} (Rp {total_cost_usd * USD_TO_IDR:.2f})")
print(f"Avg cost per ticket      : ${avg_cost_usd:.6f} (Rp {avg_cost_usd * USD_TO_IDR:.4f})")

# Target check
print()
print("Target Compliance:")
print(f"  Cost < $0.01/ticket    : {'PASS' if avg_cost_usd < 0.01 else 'FAIL'} (avg ${avg_cost_usd:.6f})")
print(f"  Latency P50 < 4s       : {'PASS' if lat_p50 < 4.0 else 'FAIL'} ({lat_p50:.2f}s)")
print(f"  Latency P95 < 4s       : {'PASS' if lat_p95 < 4.0 else 'FAIL'} ({lat_p95:.2f}s)")

BATCH PROCESSING SUMMARY
Total tickets processed  : 150
Errors                   : 0
Classification accuracy  : 86.0% (on 150 labeled tickets)
Latency P50              : 5.10s
Latency P95              : 12.03s
Total cost (all tickets) : $0.168942 (Rp 2787.54)
Avg cost per ticket      : $0.001126 (Rp 18.5836)

Target Compliance:
  Cost < $0.01/ticket    : PASS (avg $0.001126)
  Latency P50 < 4s       : FAIL (5.10s)
  Latency P95 < 4s       : FAIL (12.03s)


In [24]:
# Category distribution analysis
print("Predicted Category Distribution:")
print(out_df["predicted_category"].value_counts().to_string())
print()

if "true_category" in out_df.columns:
    print("True vs Predicted (first 20):")
    comp_df = out_df[["ticket_id", "ticket_text", "true_category", "predicted_category", "correct"]].head(20)
    print(comp_df.to_string(index=False))

Predicted Category Distribution:
predicted_category
hardware_request    21
erp_sap             20
email_issue         16
mfa_reset           16
password_reset      14
software_request    14
offboarding         12
wifi_network        11
onboarding          10
vpn_issue            9
out_of_scope         7

True vs Predicted (first 20):
    ticket_id                          ticket_text    true_category predicted_category  correct
TKT-2605-0001        Immediate offboarding request      offboarding        offboarding        1
TKT-2605-0002                Locked out of SAP P01          erp_sap     password_reset        0
TKT-2605-0003            Convert to shared mailbox      offboarding        email_issue        0
TKT-2605-0004       Account creation for new staff       onboarding         onboarding        1
TKT-2605-0005                Locked out of SAP P01          erp_sap     password_reset        0
TKT-2605-0006         Install MS Visio for project software_request   software_request  

## 10. Step 4 - Evaluation (gpt-4o)

Evaluate quality using a higher-tier model to avoid bias. Metrics:
- **Groundedness** (1-5): Does the answer stick to retrieved SOP context?
- **Relevance** (1-5): Does the answer address the actual ticket?
- **Coherence** (1-5): Is the response readable and professional?

In [25]:
EVALUATOR_SYSTEM = """You are an impartial quality evaluator for an IT helpdesk AI assistant.
You will score generated responses on three dimensions.

Return your evaluation as a JSON object with exactly these keys:
{
  "groundedness": <integer 1-5>,
  "relevance": <integer 1-5>,
  "coherence": <integer 1-5>,
  "groundedness_reason": "<one sentence>",
  "relevance_reason": "<one sentence>",
  "coherence_reason": "<one sentence>"
}

Scoring rubrics:

GROUNDEDNESS (Does the response stay grounded in the provided SOP context?):
  5 = All steps traceable to SOP context, no invented procedures
  4 = Mostly grounded, minor elaboration acceptable
  3 = Some steps from SOP, some generic IT knowledge added
  2 = Significant content not from SOP context
  1 = Response largely hallucinated or ignores SOP context

RELEVANCE (Does the response address what the ticket is actually asking?):
  5 = Directly and completely addresses the ticket issue
  4 = Mostly relevant, minor tangent
  3 = Partially addresses the ticket
  2 = Loosely related to the issue
  1 = Does not address the ticket issue

COHERENCE (Is the response clear, professional, and well-structured?):
  5 = Excellent: clear steps, professional tone, good structure
  4 = Good: readable with minor issues
  3 = Average: understandable but awkward
  2 = Poor: confusing or unprofessional
  1 = Very poor: incoherent

Return ONLY the JSON object. No preamble, no explanation outside the JSON."""


def evaluate_response(ticket_text: str, sop_context: str, response: str) -> tuple:
    """
    Evaluate a generated response using gpt-4o.
    Returns (scores_dict, cost_dict).
    """
    user_prompt = f"""## Ticket
{ticket_text}

## SOP Context Provided to Generator
{sop_context[:2000] if sop_context else 'No SOP context provided.'}

## Generated Response
{response}

Evaluate the generated response on groundedness, relevance, and coherence."""

    try:
        resp = client.chat.completions.create(
            model=EVALUATOR_MODEL,
            messages=[
                {"role": "system", "content": EVALUATOR_SYSTEM},
                {"role": "user",   "content": user_prompt},
            ],
            temperature=0,
            max_tokens=400,
            response_format={"type": "json_object"},
        )
        cost = estimate_cost(EVALUATOR_MODEL, resp.usage.prompt_tokens, resp.usage.completion_tokens)
        scores = json.loads(resp.choices[0].message.content)
        return scores, cost
    except Exception as e:
        zero_cost = estimate_cost(EVALUATOR_MODEL, 0, 0)
        return {"groundedness": 0, "relevance": 0, "coherence": 0,
                "groundedness_reason": str(e), "relevance_reason": "", "coherence_reason": ""}, zero_cost


# Quick eval test
sample_resp, _ = generate_response(sample_ticket, sample_category, chunks)
sample_sop_context = "\n".join(c["content"] for c in chunks)
test_scores, test_cost = evaluate_response(sample_ticket, sample_sop_context, sample_resp)

print("=== Evaluator Test ===")
print(f"Ticket   : {sample_ticket}")
print(f"Category : {sample_category}")
print()
print(f"Groundedness : {test_scores.get('groundedness', 'N/A')}/5 - {test_scores.get('groundedness_reason', '')}")
print(f"Relevance    : {test_scores.get('relevance', 'N/A')}/5 - {test_scores.get('relevance_reason', '')}")
print(f"Coherence    : {test_scores.get('coherence', 'N/A')}/5 - {test_scores.get('coherence_reason', '')}")
print(f"Eval cost    : ${test_cost['cost_usd']:.6f}")

=== Evaluator Test ===
Ticket   : Locked out of SAP P01 after too many failed login attempts
Category : erp_sap

Groundedness : 5/5 - The response strictly follows the SOP-004 procedures for unlocking an SAP account and resetting the password.
Relevance    : 5/5 - The response directly addresses the issue of being locked out of SAP P01 due to failed login attempts.
Coherence    : 5/5 - The response is clear, professional, and well-structured, providing step-by-step instructions.
Eval cost    : $0.004370


In [26]:
# Run evaluation on EVAL_SAMPLE_SIZE tickets
# Filter to tickets that were processed successfully with a draft response
eval_pool = out_df[
    (out_df["draft_response"].notna()) &
    (out_df["draft_response"] != "") &
    (~out_df["predicted_category"].str.startswith("error", na=False)) &
    (out_df["predicted_category"] != "out_of_scope")
].copy()

# Sample evenly across categories if possible
sample_size = min(EVAL_SAMPLE_SIZE, len(eval_pool))
if sample_size < len(eval_pool):
    eval_sample = eval_pool.groupby("predicted_category", group_keys=False).apply(
        lambda x: x.sample(min(len(x), max(1, sample_size // eval_pool["predicted_category"].nunique())))
    ).head(sample_size)
else:
    eval_sample = eval_pool

print(f"Running evaluation on {len(eval_sample)} tickets using {EVALUATOR_MODEL}...")

eval_results = []
eval_total_cost = 0.0

for _, row in tqdm(eval_sample.iterrows(), total=len(eval_sample), desc="Evaluating"):
    ticket_text   = str(row.get("ticket_text", ""))
    draft         = str(row.get("draft_response", ""))
    sources       = str(row.get("retrieved_sources", ""))

    # Rebuild SOP context from retrieved sources
    if sources:
        sop_ids = [s.strip() for s in sources.split("|")]
        context_chunks = [c["content"] for c in sop_chunks if c["sop_id"] in sop_ids]
        sop_ctx = "\n\n".join(context_chunks)
    else:
        sop_ctx = ""

    scores, cost = evaluate_response(ticket_text, sop_ctx, draft)
    eval_total_cost += cost["cost_usd"]

    eval_results.append({
        "ticket_id":            row.get("ticket_id", ""),
        "ticket_text":          ticket_text[:100],
        "predicted_category":   row.get("predicted_category", ""),
        "retrieved_sources":    sources,
        "groundedness":         scores.get("groundedness", 0),
        "relevance":            scores.get("relevance", 0),
        "coherence":            scores.get("coherence", 0),
        "groundedness_reason":  scores.get("groundedness_reason", ""),
        "relevance_reason":     scores.get("relevance_reason", ""),
        "coherence_reason":     scores.get("coherence_reason", ""),
    })
    time.sleep(0.3)


eval_df = pd.DataFrame(eval_results)
eval_df.to_csv(EVAL_CSV, index=False)
print(f"\nEvaluation complete. Saved to {EVAL_CSV}")
print(f"Evaluation cost: ${eval_total_cost:.6f} (Rp {eval_total_cost * USD_TO_IDR:.2f})")

Running evaluation on 20 tickets using gpt-4o...


Evaluating: 100%|██████████| 20/20 [00:59<00:00,  2.98s/it]


Evaluation complete. Saved to eval_report.csv
Evaluation cost: $0.085260 (Rp 1406.79)


In [27]:
# Evaluation summary report
print("=" * 60)
print("EVALUATION SUMMARY REPORT")
print(f"Model used for evaluation: {EVALUATOR_MODEL}")
print(f"Tickets evaluated: {len(eval_df)}")
print("=" * 60)

metrics = ["groundedness", "relevance", "coherence"]
for m in metrics:
    vals = eval_df[m]
    avg  = vals.mean()
    mn   = vals.min()
    mx   = vals.max()
    pct  = (avg / 5) * 100
    print(f"{m.capitalize():<15}: avg={avg:.2f}/5 ({pct:.0f}%)  min={mn}  max={mx}")

print()
# Target metrics from task.md
gnd_pct = (eval_df["groundedness"].mean() / 5) * 100
rel_pct = (eval_df["relevance"].mean() / 5) * 100

print("Target Compliance (task.md):")
print(f"  Groundedness >= 80%  : {'PASS' if gnd_pct >= 80 else 'FAIL'} ({gnd_pct:.1f}%)")
print(f"  Relevance >= 70%     : {'PASS' if rel_pct >= 70 else 'FAIL'} ({rel_pct:.1f}%)")

print()
print("Score Distribution:")
for m in metrics:
    dist = eval_df[m].value_counts().sort_index()
    print(f"  {m}: " + " | ".join(f"{k}:{v}" for k, v in dist.items()))

print()
print("Per-Category Average Scores:")
cat_scores = eval_df.groupby("predicted_category")[["groundedness", "relevance", "coherence"]].mean().round(2)
print(cat_scores.to_string())

EVALUATION SUMMARY REPORT
Model used for evaluation: gpt-4o
Tickets evaluated: 20
Groundedness   : avg=2.95/5 (59%)  min=1  max=5
Relevance      : avg=4.05/5 (81%)  min=2  max=5
Coherence      : avg=4.70/5 (94%)  min=4  max=5

Target Compliance (task.md):
  Groundedness >= 80%  : FAIL (59.0%)
  Relevance >= 70%     : PASS (81.0%)

Score Distribution:
  groundedness: 1:6 | 2:4 | 3:1 | 4:3 | 5:6
  relevance: 2:3 | 3:4 | 4:2 | 5:11
  coherence: 4:6 | 5:14

Per-Category Average Scores:
                    groundedness  relevance  coherence
predicted_category                                    
email_issue                  2.5        4.0        5.0
erp_sap                      2.5        4.0        5.0
hardware_request             3.0        4.0        5.0
mfa_reset                    5.0        5.0        5.0
offboarding                  1.0        3.5        4.5
onboarding                   4.5        5.0        5.0
password_reset               2.0        3.5        4.0
software_request  

In [28]:
# Show low-scoring responses for analysis
low_threshold = 3
low_scores = eval_df[
    (eval_df["groundedness"] <= low_threshold) |
    (eval_df["relevance"] <= low_threshold)
]

print(f"Low-scoring tickets (groundedness or relevance <= {low_threshold}): {len(low_scores)}")
if len(low_scores) > 0:
    for _, row in low_scores.iterrows():
        print(f"\n  Ticket : {row['ticket_id']} - {row['ticket_text'][:80]}")
        print(f"  Category: {row['predicted_category']}")
        print(f"  Scores  : G={row['groundedness']} R={row['relevance']} C={row['coherence']}")
        print(f"  Reason  : {row['groundedness_reason']}")

Low-scoring tickets (groundedness or relevance <= 3): 11

  Ticket : TKT-2605-0038 - Convert to shared mailbox
  Category: email_issue
  Scores  : G=3 R=5 C=5
  Reason  : The response correctly identifies that the SOP does not cover converting to a shared mailbox, but it does not provide any specific steps from the SOP context.

  Ticket : TKT-2605-0077 - Convert to shared mailbox
  Category: email_issue
  Scores  : G=2 R=3 C=5
  Reason  : The response does not reference any specific steps from the provided SOP context related to converting a mailbox to a shared mailbox.

  Ticket : TKT-2605-0022 - Cannot access finance T-Code
  Category: erp_sap
  Scores  : G=1 R=3 C=5
  Reason  : The response is not grounded in the provided SOP context, which is about VPN troubleshooting, not SAP T-Code access.

  Ticket : TKT-2605-0032 - Cannot print from SAP
  Category: hardware_request
  Scores  : G=1 R=3 C=5
  Reason  : The response is not grounded in the provided SOP context, which is about SAP 

## 11. Cost & Performance Report

In [29]:
print("=" * 70)
print("COST & PERFORMANCE REPORT")
print("BSI IT Service Desk AI Assistant - Mini Project 5")
print("=" * 70)

print()
print("MODEL CHOICES & JUSTIFICATION")
print("-" * 40)
model_table = [
    ("Classification", CLASSIFIER_MODEL, "Fast, cheap routing. Accuracy sufficient for category assignment."),
    ("Response Gen.",  RESPONDER_MODEL,  "Balanced quality/cost. Better than nano for complex SOP responses."),
    ("Evaluation",     EVALUATOR_MODEL,  "High-tier model to avoid bias in quality scoring (task.md requirement)."),
    ("Embedding",      EMBEDDING_MODEL,  "Cost-efficient, high-quality vectors for semantic retrieval."),
]
print(f"{'Step':<18} {'Model':<22} {'Rationale'}")
print("-" * 70)
for step, model, reason in model_table:
    print(f"{step:<18} {model:<22} {reason}")

print()
print("TOKEN PRICING (per 1M tokens)")
print("-" * 40)
print(f"{'Model':<28} {'Input':<12} {'Output'}")
print("-" * 50)
for model, price in PRICING.items():
    print(f"{model:<28} ${price['input']:.2f}/M       ${price['output']:.2f}/M")

print()
print("PIPELINE PERFORMANCE (batch results)")
print("-" * 40)
if "cost_total_usd" in out_df.columns and len(out_df) > 0:
    valid = out_df[out_df["predicted_category"] != "error"]
    print(f"  Tickets processed    : {len(out_df)}")
    print(f"  Total cost           : ${valid['cost_total_usd'].sum():.6f} (Rp {valid['cost_total_usd'].sum() * USD_TO_IDR:.2f})")
    print(f"  Avg cost/ticket      : ${valid['cost_total_usd'].mean():.6f} (Rp {valid['cost_total_usd'].mean() * USD_TO_IDR:.4f})")
    print(f"  Max cost/ticket      : ${valid['cost_total_usd'].max():.6f}")
    if "latency_total_s" in valid.columns:
        print(f"  Latency P50          : {valid['latency_total_s'].quantile(0.50):.2f}s")
        print(f"  Latency P95          : {valid['latency_total_s'].quantile(0.95):.2f}s")
        print(f"  Latency max          : {valid['latency_total_s'].max():.2f}s")
    print(f"  Cost target (<$0.01) : {'PASS' if valid['cost_total_usd'].mean() < 0.01 else 'FAIL'}")

print()
print("RETRIEVAL CONFIGURATION")
print("-" * 40)
print(f"  Method             : In-memory cosine similarity (text-embedding-3-small)")
print(f"  Chunk size         : ~500 tokens with 50 token overlap")
print(f"  Top-K              : {SEARCH_TOP_K}")
print(f"  Total SOP chunks   : {len(sop_chunks)}")
print(f"  Region             : Local VSCode (no Azure dependency)")

print()
print("HONESTY GUARDRAIL")
print("-" * 40)
print("  out_of_scope tickets: Redirected to appropriate department (C3 compliance)")
print("  No-SOP fallback: Escalation to IT Level 2 (never hallucinated)")
print("  Citations: Every response ends with [Sumber SOP: <ID>] (C4 compliance)")

COST & PERFORMANCE REPORT
BSI IT Service Desk AI Assistant - Mini Project 5

MODEL CHOICES & JUSTIFICATION
----------------------------------------
Step               Model                  Rationale
----------------------------------------------------------------------
Classification     gpt-4.1-nano           Fast, cheap routing. Accuracy sufficient for category assignment.
Response Gen.      gpt-4.1-mini           Balanced quality/cost. Better than nano for complex SOP responses.
Evaluation         gpt-4o                 High-tier model to avoid bias in quality scoring (task.md requirement).
Embedding          text-embedding-3-small Cost-efficient, high-quality vectors for semantic retrieval.

TOKEN PRICING (per 1M tokens)
----------------------------------------
Model                        Input        Output
--------------------------------------------------
gpt-4.1-nano                 $0.10/M       $0.40/M
gpt-4.1-mini                 $0.40/M       $1.60/M
gpt-4o               

## 12. Final Demo - 5 Representative Tickets

In [31]:
demo_tickets = [
    ("DEMO-001", "Locked out of SAP P01 system after 3 failed login attempts"),
    ("DEMO-002", "Cannot connect to MITSUBISHI-CORP wifi, error: Can't connect to this network"),
    ("DEMO-003", "New phone, need to setup MFA Microsoft Authenticator again"),
    ("DEMO-004", "Immediate offboarding request for resigned employee, please disable all accounts"),
    ("DEMO-005", "Request to install AutoCAD 2024 for engineering team project"),
]

print("=" * 70)
print("FINAL DEMO - 5 REPRESENTATIVE TICKETS")
print("=" * 70)

demo_results = []
for tid, text in demo_tickets:
    print(f"\n{'='*70}")
    print(f"Ticket ID   : {tid}")
    print(f"Description : {text}")
    print("-" * 70)

    result = run_pipeline(tid, text, verbose=True)
    demo_results.append(result)

    print("\nDRAFT RESPONSE:")
    print("-" * 70)
    print(result["draft_response"])

print("\n" + "=" * 70)
print("DEMO COMPLETE")
print("=" * 70)

demo_costs = [r["cost_total_usd"] for r in demo_results]
demo_lats  = [r["latency_total_s"] for r in demo_results]
print(f"Demo avg cost/ticket : ${sum(demo_costs)/len(demo_costs):.6f}")
print(f"Demo avg latency     : {sum(demo_lats)/len(demo_lats):.2f}s")

FINAL DEMO - 5 REPRESENTATIVE TICKETS

Ticket ID   : DEMO-001
Description : Locked out of SAP P01 system after 3 failed login attempts
----------------------------------------------------------------------
  Category  : password_reset (6.891s)
  Chunks    : 3 retrieved (0.391s)
  Response  : 1520 chars (6.941s)
  Cost      : $0.001349 / Rp 22.2569

DRAFT RESPONSE:
----------------------------------------------------------------------
Yth. Bapak/Ibu,

Terima kasih telah menghubungi BSI Helpdesk terkait kendala terkunci pada sistem SAP P01 setelah 3 kali percobaan login gagal. Berikut langkah-langkah yang akan kami lakukan untuk membuka kembali akses Anda:

1. Kami akan membuka aplikasi **SAP Logon** dan masuk ke sistem P01 (Production).
2. Jalankan T-Code: **SU01**.
3. Masukkan SAP User ID Anda.
4. Klik ikon **Lock/Unlock** (gambar gembok) untuk memeriksa status akun.
5. Jika akun terkunci karena kesalahan login berulang, kami akan klik tombol **Unlock** untuk membuka kunci akun Anda.
6